# SparseBinaryDecomposition

[Codility task](https://app.codility.com/programmers/trainings/9/sparse_binary_decomposition/) - Exercise 9, Bitwise operations - *hard*

`N` is **sparse** if its binary form has no two consecutive 1s. `P` and `Q` are a **sparse decomposition** of `N` if both are sparse and `P + Q = N`.

`solution(N)` returns any one part of a sparse decomposition of `N`, or `-1` if none exists. Example: `N = 26` -> `8`, `9`, `17` or `18` (also `5`, `10`, `16`, `21`).

Assumption: `N` is in `[0..1,000,000,000]`.

## Idea: digit DP over the binary addition

Build `P` and `Q` from the least significant bit upward, simulating the binary addition `P + Q`:

- at bit `i` choose `p_i, q_i` in `{0, 1}`;
- the sum bit `(p_i + q_i + carry) % 2` must equal bit `i` of `N`;
- the new carry is `(p_i + q_i + carry) // 2`;
- sparseness: `p_i` and `p_{i-1}` cannot both be 1 (same for `q`).

State: `(i, previous p bit, previous q bit, carry)`, 8 combinations per position, 32 positions. With memoisation each state is visited once, so the cost is O(log N). At the end we need `carry == 0`. The recursion returns the bits of `P` above position `i`, or `None` if the state is a dead end.

In [ ]:
from functools import lru_cache

BITS = 32     # N < 2**30, and P, Q <= N, so 32 bits are more than enough


def solution(N):
    @lru_cache(maxsize=None)
    def go(i, prev_p, prev_q, carry):
        """Bits of P from position i upward (as an int shifted by i), or None."""
        if i == BITS:
            return 0 if carry == 0 else None
        n_bit = (N >> i) & 1
        for p in (0, 1):
            if p and prev_p:
                continue
            for q in (0, 1):
                if q and prev_q:
                    continue
                total = p + q + carry
                if total & 1 != n_bit:
                    continue
                rest = go(i + 1, p, q, total >> 1)
                if rest is not None:
                    return p | (rest << 1)
        return None

    P = go(0, 0, 0, 0)
    return -1 if P is None else P

## Validator

The answer is not unique, so tests must check the **properties** of the returned value rather than compare it with one number.

In [ ]:
def is_sparse(x):
    return x & (x >> 1) == 0


def is_valid_part(N, P):
    """P is one part of a sparse decomposition of N."""
    return 0 <= P <= N and is_sparse(P) and is_sparse(N - P)


assert is_sparse(41) and not is_sparse(26)
assert is_valid_part(26, 8) and is_valid_part(26, 9)
assert not is_valid_part(26, 2)        # 24 = 11000 is not sparse
print("validator OK")

## Official example

In [ ]:
P = solution(26)
print("N = 26 ->", P, "and", 26 - P)
assert P in {5, 8, 9, 10, 16, 17, 18, 21}
assert is_valid_part(26, P)

## Edge cases

In [ ]:
for N in (0, 1, 2, 3, 7, 15, 2**30 - 1, 10**9):
    P = solution(N)
    assert P != -1, N
    assert is_valid_part(N, P), (N, P)
print("edge cases OK")

## Brute-force check

For every `N` below a bound, compute by enumeration whether a decomposition exists, and compare:

- if one exists, `solution` must return a valid part;
- if none exists, `solution` must return `-1`.

In [ ]:
LIMIT = 3000
sparse_numbers = [x for x in range(LIMIT) if is_sparse(x)]
sparse_set = set(sparse_numbers)

no_decomposition = []
for N in range(LIMIT):
    exists = any((N - p) in sparse_set for p in sparse_numbers if p <= N)
    P = solution(N)
    if exists:
        assert P != -1 and is_valid_part(N, P), (N, P)
    else:
        assert P == -1, (N, P)
        no_decomposition.append(N)

print(f"checked N = 0..{LIMIT - 1}; N without a sparse decomposition: {no_decomposition}")

Every integer tested has a decomposition (the list above is empty): it is a known property that any non-negative integer is a sum of two sparse integers. The `-1` branch is kept because the task allows it; the brute force confirms that it never triggers wrongly.

## Large and random inputs, and timing

In [ ]:
import random
import time

random.seed(1)
cases = [10**9, 10**9 - 1, 2**29, 2**29 - 1, 2**30 - 1] + [random.randrange(10**9 + 1) for _ in range(2000)]

start = time.perf_counter()
for N in cases:
    P = solution(N)
    assert P != -1 and is_valid_part(N, P), (N, P)
elapsed = time.perf_counter() - start
print(f"{len(cases)} large cases OK in {elapsed:.3f}s")

## Notes

- A digit DP is the natural tool when a constraint couples **adjacent bits** and a carry links positions.
- `lru_cache` memoises `(i, prev_p, prev_q, carry)`: at most `32 * 8 = 256` states per call, so O(log N).
- `Q` is not needed explicitly: `Q = N - P`. The validator checks that it is sparse too.
- Possible extension: return `(P, Q)` and compare with the number of decompositions (count instead of find).